# 07 - SHAP Explainability & Model Interpretation

This notebook generates global and local SHAP (SHapley Additive exPlanations) for the tuned Random Forest model trained on the CIC-MalMem-2022 dataset with 30 selected features.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

os.makedirs("../reports/figures", exist_ok=True)

In [ ]:
# Load Tuned Model and Test Features
model = joblib.load("../models/best_model.pkl")
feature_names = joblib.load("../models/feature_names.pkl")

X_test = joblib.load("../data/processed/X_test.pkl")
X_test_df = pd.DataFrame(X_test, columns=feature_names)
print(f"Loaded test dataset with {X_test_df.shape[0]} samples and {X_test_df.shape[1]} features.")

In [ ]:
# Initialize SHAP TreeExplainer
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_df)

if isinstance(shap_values, list):
    vals = shap_values[1]  # Positive class (Ransomware)
    expected_val = explainer.expected_value[1]
elif isinstance(shap_values, np.ndarray) and len(shap_values.shape) == 3:
    vals = shap_values[:, :, 1]
    expected_val = explainer.expected_value[1]
else:
    vals = shap_values
    expected_val = explainer.expected_value

In [ ]:
# 1. Global Feature Importance Bar Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(vals, X_test_df, plot_type="bar", show=False)
plt.title("Global Feature Importance (SHAP Bar Plot)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("../reports/figures/shap_bar.png", dpi=300)
plt.show()

In [ ]:
# 2. SHAP Beeswarm Plot
plt.figure(figsize=(10, 7))
shap.summary_plot(vals, X_test_df, show=False)
plt.title("SHAP Beeswarm Plot - Tuned Random Forest", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("../reports/figures/shap_beeswarm.png", dpi=300)
plt.show()

In [ ]:
# 3. Local Explanation (Waterfall Plot)
sample_idx = 0
exp = shap.Explanation(
    values=vals[sample_idx],
    base_values=expected_val if np.isscalar(expected_val) else expected_val[0],
    data=X_test_df.iloc[sample_idx].values,
    feature_names=feature_names
)
shap.plots.waterfall(exp, show=False)
plt.tight_layout()
plt.savefig("../reports/figures/shap_waterfall.png", dpi=300)
plt.show()